# 02e — Building-level pedestrian accessibility: Lordelo do Ouro e Massarelos

This notebook reproduces the parish-level pedestrian-network stage used to construct the municipal building-level accessibility dataset.

**Fixed analytical specification**

- 800 m pedestrian-network catchment;
- directed pedestrian graph;
- 2 m DTM;
- raster-directional slope;
- normalised Tobler adjustment;
- reference walking speed of 0.7 m/s;
- additional travel-time outputs for 0.5 and 0.9 m/s;
- seven service and urban-amenity categories;
- origin–destination pairs retained within 800 m;
- topology quality control performed before the final parish export.

The municipal 10/15/20-minute sensitivity analysis is performed only after all seven parish outputs have been integrated in Notebook 03.


## Repository paths and outputs

Raw inputs are read from `data/raw/`.

All temporary and parish-level outputs are isolated under:

`data/intermediate/parish_accessibility/<parish_slug>/`

The notebook does not write generated CSV files into the `notebooks/` directory.


## 1. Load and validate the parish inputs

In [ ]:
from pathlib import Path
import os
import json
import pandas as pd
import geopandas as gpd
from shapely import wkt
from shapely.geometry import shape

def resolve_repo_root():
    cwd = Path.cwd().resolve()
    if cwd.name == 'notebooks':
        return cwd.parent
    if (cwd / 'notebooks').exists() and (cwd / 'data').exists():
        return cwd
    for parent in cwd.parents:
        if (parent / 'notebooks').exists() and (parent / 'data').exists():
            return parent
    raise RuntimeError('Repository root not found. Run the notebook from the repository root or from the notebooks directory.')
REPO_ROOT = resolve_repo_root()
RAW_DATA = REPO_ROOT / 'data' / 'raw'
INTERMEDIATE_DATA = REPO_ROOT / 'data' / 'intermediate'
PARISH_NAME = 'Lordelo do Ouro e Massarelos'
PARISH_SLUG = 'lordelo_ouro_massarelos'
PARISH_WORK_DIR = INTERMEDIATE_DATA / 'parish_accessibility' / PARISH_SLUG
PARISH_WORK_DIR.mkdir(parents=True, exist_ok=True)
BUILDINGS_FILE = RAW_DATA / 'edificios.csv'
SERVICES_FILE = RAW_DATA / 'servicos.csv'
PARISHES_FILE = RAW_DATA / 'Workflow validation diagnostic.'
NETWORK_FILE = RAW_DATA / 'rede_final_ligada.csv'
DTM_FILE = RAW_DATA / 'MDT_Porto_clip.tif'
for required_file in [BUILDINGS_FILE, SERVICES_FILE, PARISHES_FILE, NETWORK_FILE, DTM_FILE]:
    if not required_file.exists():
        raise FileNotFoundError(f'Missing required input: {required_file}')

def load_csv_with_geometry(path, possible_geom_cols=('geometry_wkt', 'geometry')):
    df = pd.read_csv(path, dtype=str, low_memory=False)
    geom_col = next((c for c in possible_geom_cols if c in df.columns), None)
    if geom_col is None:
        return df
    df['geometry'] = df[geom_col].apply(lambda value: wkt.loads(value) if pd.notna(value) and str(value).strip() else None)
    gdf = gpd.GeoDataFrame(df, geometry='geometry', crs='EPSG:4326')
    return gdf.dropna(subset=['geometry'])

def safe_geojson_loads(value):
    try:
        return shape(json.loads(value))
    except (ValueError, TypeError, json.JSONDecodeError):
        return None
buildings = load_csv_with_geometry(BUILDINGS_FILE)
services = pd.read_csv(SERVICES_FILE, dtype=str, low_memory=False)
parishes = pd.read_csv(PARISHES_FILE, sep=';', encoding='latin1')
network = load_csv_with_geometry(NETWORK_FILE)
if not isinstance(network, gpd.GeoDataFrame):
    raise ValueError('The pedestrian-network file does not contain a recognised geometry column.')
if not isinstance(buildings, gpd.GeoDataFrame):
    raise ValueError('The building file does not contain a recognised geometry column.')
parishes['geometry'] = parishes['Geo Shape'].apply(safe_geojson_loads)
parishes_gdf = gpd.GeoDataFrame(parishes, geometry='geometry', crs='EPSG:4326').dropna(subset=['geometry'])
selected_parishes = parishes_gdf[parishes_gdf['Official Name Parish'].str.contains(PARISH_NAME, case=False, na=False, regex=False)].copy()
if selected_parishes.empty:
    raise ValueError(f'Parish not found: {PARISH_NAME}')
parish_geometry = selected_parishes.geometry.union_all()
selected_buildings = buildings[buildings.geometry.within(parish_geometry)].copy()
services['Longitude'] = pd.to_numeric(services['Longitude'], errors='coerce')
services['Latitude'] = pd.to_numeric(services['Latitude'], errors='coerce')
services = services.dropna(subset=['Longitude', 'Latitude']).copy()
services_gdf = gpd.GeoDataFrame(services, geometry=gpd.points_from_xy(services['Longitude'], services['Latitude']), crs='EPSG:4326')
parish_projected = selected_parishes.to_crs(3763)
services_projected = services_gdf.to_crs(3763)
buildings_projected = selected_buildings.to_crs(3763)
network_projected = network.to_crs(3763)
parish_geometry_projected = parish_projected.geometry.union_all()
buffer_800m = parish_geometry_projected.buffer(800)
services_buffer_800m = services_projected[services_projected.geometry.within(buffer_800m)].copy()
selected_network = network_projected[network_projected.geometry.intersects(buffer_800m)].copy()
network_union = selected_network.geometry.union_all()
buildings_projected['dist_to_network'] = buildings_projected['geometry'].apply(lambda geom: geom.centroid.distance(network_union) if geom.geom_type in ['Polygon', 'MultiPolygon'] else geom.distance(network_union))
buildings_projected['ligado_rede'] = buildings_projected['dist_to_network'] <= 60
print('Parish:', PARISH_NAME)
print('Buildings:', len(buildings_projected))
print('Services within the 800 m parish buffer:', len(services_buffer_800m))
print('Network segments within the buffered area:', len(selected_network))
print('Buildings connected to the pedestrian network:', int(buildings_projected['ligado_rede'].sum()))
ORIGINAL_WORKING_DIRECTORY = Path.cwd().resolve()
os.chdir(PARISH_WORK_DIR)
print('Intermediate parish directory:', PARISH_WORK_DIR)

## 2. Build the directed pedestrian graph

In [ ]:
import networkx as nx
from shapely.geometry import LineString
G_total = nx.DiGraph()
for _, row in selected_network.iterrows():
    geom = row.geometry
    if geom is None:
        continue

    def add_segments(line):
        coords = list(line.coords)
        for i in range(len(coords) - 1):
            u = coords[i]
            v = coords[i + 1]
            length = LineString([u, v]).length
            if length <= 0:
                continue
            G_total.add_edge(u, v, length=float(length))
            G_total.add_edge(v, u, length=float(length))
    if geom.geom_type == 'LineString':
        add_segments(geom)
    elif geom.geom_type == 'MultiLineString':
        for line in geom.geoms:
            add_segments(line)
print('Directed graph created.')
print('Nodes:', G_total.number_of_nodes())
print('Directed edges:', G_total.number_of_edges())
assert G_total.number_of_nodes() > 0 and G_total.number_of_edges() > 0

## 3. Derive raster-directional slope from the 2 m DTM

In [ ]:
import rasterio
import numpy as np
import math
dtm_path = DTM_FILE
dtm_src = rasterio.open(dtm_path)
print('MDT loaded.')
print('CRS:', dtm_src.crs)
print('Resolution:', dtm_src.res)
if str(dtm_src.crs) != 'EPSG:3763':
    raise ValueError(f'CRS inesperado no MDT: {dtm_src.crs}. Esperado EPSG:3763.')
dem_ma = dtm_src.read(1, masked=True).astype('float64')
dem = dem_ma.filled(np.nan)
res_x = abs(float(dtm_src.transform.a))
res_y = abs(float(dtm_src.transform.e))
if res_x <= 0 or res_y <= 0:
    raise ValueError('Resolution inválida no MDT.')
grad_row, grad_col = np.gradient(dem, res_y, res_x)
grad_x = grad_col
grad_y = -grad_row
DEM_RES_M = max(res_x, res_y)

def _sample_array_nearest(arr, x, y, src=dtm_src):
    """Amostra nearest-neighbour de uma matriz alinhada com o MDT."""
    try:
        row, col = src.index(float(x), float(y))
        if row < 0 or col < 0 or row >= arr.shape[0] or (col >= arr.shape[1]):
            return np.nan
        val = arr[row, col]
        return float(val) if np.isfinite(val) else np.nan
    except Exception:
        return np.nan

def directional_raster_slope(u, v, length=None):
    """Terrain-adjustment validation diagnostic."""
    x1, y1 = (float(u[0]), float(u[1]))
    x2, y2 = (float(v[0]), float(v[1]))
    if length is None:
        length = math.hypot(x2 - x1, y2 - y1)
    length = float(length)
    if not np.isfinite(length) or length <= 0:
        return 0.0
    ux = (x2 - x1) / length
    uy = (y2 - y1) / length
    n_samples = max(1, int(math.ceil(length / DEM_RES_M)))
    vals = []
    for i in range(n_samples):
        t = (i + 0.5) / n_samples
        x = x1 + t * (x2 - x1)
        y = y1 + t * (y2 - y1)
        gx = _sample_array_nearest(grad_x, x, y)
        gy = _sample_array_nearest(grad_y, x, y)
        if np.isfinite(gx) and np.isfinite(gy):
            vals.append(gx * ux + gy * uy)
    if not vals:
        return 0.0
    return float(np.mean(vals))

def assign_raster_slopes_to_graph(G, only_missing=False):
    """Terrain-adjustment validation diagnostic."""
    n = 0
    for u, v, data in G.edges(data=True):
        if only_missing and 'raster_slope' in data and np.isfinite(data['raster_slope']):
            continue
        length = float(data.get('length', 0.0) or 0.0)
        if length <= 0:
            data['raster_slope'] = 0.0
        else:
            data['raster_slope'] = directional_raster_slope(u, v, length)
        n += 1
    return n
n_slopes = assign_raster_slopes_to_graph(G_total)
print(f'Terrain-adjustment validation diagnostic.{n_slopes}Workflow validation diagnostic.')
_slopes = np.array([d.get('raster_slope', np.nan) for _, _, d in G_total.edges(data=True)], dtype=float)
_slopes = _slopes[np.isfinite(_slopes)]
if len(_slopes):
    print('Absolute raster-directional slope (%):')
    for p in [50, 90, 95, 99, 99.5, 99.9, 100]:
        print(f'  P{p:g}: {np.percentile(np.abs(_slopes) * 100, p):.2f}%')

## Fixed accessibility specification

The 800 m catchment is defined by pedestrian-network distance. At 0.7 m/s, 800 m corresponds to about 19 minutes on level terrain, but this value is used only as an interpretative equivalent and not as a temporal filter.

Flat and Tobler-adjusted travel times are stored for 0.5, 0.7 and 0.9 m/s. Network distances, destination counts and service-category counts are calculated once. The origin–destination table retains all eligible building–destination pairs within the 800 m network catchment.


In [ ]:
import math
import heapq
import numpy as np
MAX_NETWORK_DISTANCE_M = 800.0
WALKING_SPEEDS_MS = (0.5, 0.7, 0.9)
BASELINE_WALKING_SPEED_MS = 0.7
PRIMARY_SCENARIO = 'tobler_0p7'
SAVE_OD_LONG = True
TEST_MODE = False
TEST_N_BUILDINGS_PER_SUBAREA = 100
print(f'Interpretative reference: 800 m / 0.7 m/s = {800 / 0.7 / 60:.2f} min em plano')
print('Hard travel-time threshold: NO')
print('TEST_MODE:', TEST_MODE)

def normalized_tobler_factor(slope):
    """Fator relativo da Tobler Hiking Function, normalizado para F(0)=1."""
    if slope is None or not np.isfinite(slope):
        return 1.0
    return math.exp(-3.5 * (abs(float(slope) + 0.05) - 0.05))

def tobler_unit_cost(u, v, data):
    """Terrain-adjustment validation diagnostic."""
    length = float(data.get('length', 0.0) or 0.0)
    if length <= 0:
        return 0.0
    slope = data.get('raster_slope', 0.0)
    if slope is None or not np.isfinite(slope):
        slope = 0.0
    factor = normalized_tobler_factor(float(slope))
    return length / max(factor, 1e-12)

def dijkstra_to_targets(G, source, targets, weight):
    """Workflow validation diagnostic."""
    targets = set(targets)
    if not targets:
        return {}
    if source not in G:
        return {}
    dist = {source: 0.0}
    heap = [(0.0, source)]
    settled = {}
    remaining = set(targets)
    while heap and remaining:
        d_u, u = heapq.heappop(heap)
        if u in settled:
            continue
        settled[u] = d_u
        if u in remaining:
            remaining.remove(u)
            if not remaining:
                break
        for v, edge_data in G[u].items():
            w = weight(u, v, edge_data) if callable(weight) else float(edge_data.get(weight, 1.0))
            if w is None or not np.isfinite(w) or w < 0:
                continue
            nd = d_u + float(w)
            if nd < dist.get(v, float('inf')):
                dist[v] = nd
                heapq.heappush(heap, (nd, v))
    return {t: settled[t] for t in targets if t in settled}
SCENARIO_NAMES = ['flat_0p5', 'flat_0p7', 'flat_0p9', 'tobler_0p5', 'tobler_0p7', 'tobler_0p9']
print('Stored travel-time scenarios:', SCENARIO_NAMES)
print('Downstream reference scenario:', PRIMARY_SCENARIO)

## 4. Split the parish into four computational subareas

In [ ]:
from shapely.geometry import box
minx, miny, maxx, maxy = parish_geometry_projected.bounds
mid_x = (minx + maxx) / 2
mid_y = (maxy + miny) / 2
northwest_geom = parish_geometry_projected.intersection(box(minx, mid_y, mid_x, maxy))
northeast_geom = parish_geometry_projected.intersection(box(mid_x, mid_y, maxx, maxy))
southwest_geom = parish_geometry_projected.intersection(box(minx, miny, mid_x, mid_y))
southeast_geom = parish_geometry_projected.intersection(box(mid_x, miny, maxx, mid_y))
connected_buildings_global = buildings_projected[buildings_projected['ligado_rede']].copy()
global_centroids = connected_buildings_global.geometry.centroid
connected_buildings_global['_quadrante'] = np.select([(global_centroids.x < mid_x) & (global_centroids.y >= mid_y), (global_centroids.x >= mid_x) & (global_centroids.y >= mid_y), (global_centroids.x < mid_x) & (global_centroids.y < mid_y), (global_centroids.x >= mid_x) & (global_centroids.y < mid_y)], ['northwest', 'northeast', 'southwest', 'southeast'], default='ERRO')
if (connected_buildings_global['_quadrante'] == 'ERRO').any():
    raise RuntimeError('Building-universe validation failed.')
if connected_buildings_global['osm_id'].astype(str).duplicated().any():
    raise RuntimeError('Duplicate records were detected during validation.')
print('Workflow validation diagnostic.')
print('Building-universe validation failed.', len(buildings_projected))
print('Buildings connected to the network:', len(connected_buildings_global))
print('Excluded because no valid network connection was available:', len(buildings_projected) - len(connected_buildings_global))
print('\nDistribution by computational subarea:')
print(connected_buildings_global['_quadrante'].value_counts())
assert connected_buildings_global['_quadrante'].value_counts().sum() == len(connected_buildings_global)
print('Workflow validation diagnostic.')

## 5. Prepare services and the routing functions

In [ ]:
from scipy.spatial import cKDTree
from joblib import Parallel, delayed
import pandas as pd
import networkx as nx
target_categories = ['Centro Saude', 'Farmacias', 'Hospitais', 'Supermercados', 'Bancos', 'CTT', 'Parques e jardins']
category_col = 'Category'
required_service_cols = {'Longitude', 'Latitude', category_col}
missing_service_cols = required_service_cols - set(services_buffer_800m.columns)
if missing_service_cols:
    raise ValueError(f'Service-data validation failed.{sorted(missing_service_cols)}')
unexpected_categories = sorted(set(services_buffer_800m[category_col].dropna()) - set(target_categories))
if unexpected_categories:
    print('AVISO: categorias fora das categorias-alvo (serão ignoradas):', unexpected_categories)
exact_key = [category_col, 'Longitude', 'Latitude']
if 'Name' in services_buffer_800m.columns:
    exact_key = ['Name'] + exact_key
exact_dups = services_buffer_800m.duplicated(subset=exact_key, keep=False)
if exact_dups.any():
    cols_show = [c for c in ['Name', category_col, 'Longitude', 'Latitude'] if c in services_buffer_800m.columns]
    display(services_buffer_800m.loc[exact_dups, cols_show].sort_values(cols_show))
    raise ValueError('Duplicate records were detected during validation.')
coord_cat_dups = services_buffer_800m.duplicated(subset=[category_col, 'Longitude', 'Latitude'], keep=False)
print('Candidate destinations within the buffer:', len(services_buffer_800m))
print('Registos com mesma coordenada+categoria:', int(coord_cat_dups.sum()))
if coord_cat_dups.any():
    cols_show = [c for c in ['Name', category_col, 'Longitude', 'Latitude'] if c in services_buffer_800m.columns]
    display(services_buffer_800m.loc[coord_cat_dups, cols_show].sort_values(cols_show).head(50))
    print('Workflow validation diagnostic.')
if G_total.is_directed():
    components = list(nx.weakly_connected_components(G_total))
else:
    components = list(nx.connected_components(G_total))
sorted_components = sorted(components, key=len, reverse=True)
main_component = sorted_components[0]
print(f'Total connected components: {len(components)}')
print(f'Largest component size: {len(main_component)}')
G_main = G_total.subgraph(main_component).copy()
node_list = list(G_main.nodes)
node_tree = cKDTree(np.asarray(node_list, dtype=float))

def nearest_graph_node(geom, node_tree, node_list):
    if geom.geom_type == 'Point':
        reference_point = geom
    elif geom.geom_type in ['Polygon', 'MultiPolygon']:
        reference_point = geom.centroid
    else:
        return None
    _, idx = node_tree.query((reference_point.x, reference_point.y))
    return node_list[int(idx)]

def _service_id(service):
    if 'Name' in service.index and pd.notna(service['Name']):
        return str(service['Name'])
    if 'osm_id' in service.index and pd.notna(service['osm_id']):
        return str(service['osm_id'])
    return None

def calculate_building_metrics(building_id, building_node, services_gdf, max_distance=MAX_NETWORK_DISTANCE_M):
    """Service-data validation failed."""
    try:
        dist_map = nx.single_source_dijkstra_path_length(G_main, building_node, cutoff=max_distance, weight='length')
    except (nx.NodeNotFound, nx.NetworkXError):
        dist_map = {}
    eligible_services = []
    for _, service in services_gdf.iterrows():
        node_s = service['nearest_node']
        d = dist_map.get(node_s)
        if d is None or d > max_distance:
            continue
        eligible_services.append((service, float(d)))
    target_nodes = {service['nearest_node'] for service, _ in eligible_services}
    tobler_unit_map = dijkstra_to_targets(G_main, building_node, target_nodes, tobler_unit_cost) if target_nodes else {}
    counts = {cat: 0 for cat in target_categories}
    distances = []
    od_records = []
    for service, d in eligible_services:
        cat = service[category_col]
        sid = _service_id(service)
        node_s = service['nearest_node']
        unit_tobler = tobler_unit_map.get(node_s, np.nan)
        distances.append(d)
        if cat in counts:
            counts[cat] += 1
        od_records.append({'building_id': building_id, 'service_id': sid, 'service_category': cat, 'network_distance_m': d, 'tobler_unit_cost_m': unit_tobler, 'origin_node': str(building_node), 'service_node': str(node_s)})
    if eligible_services:
        serv_min, min_distance = min(eligible_services, key=lambda x: x[1])
        nearest_service_id = _service_id(serv_min)
        nearest_service_category = serv_min[category_col]
        mean_distance = float(np.mean(distances))
    else:
        min_distance = np.nan
        nearest_service_id = None
        nearest_service_category = None
        mean_distance = np.nan
    result = {'distancia_media_servicos': mean_distance, 'distancia_minima_servico': min_distance, 'servico_min_id': nearest_service_id, 'servico_min_categoria': nearest_service_category, 'servicos_por_categoria': {k: v for k, v in counts.items() if v > 0}, 'numero_servicos_proximos': int(sum(counts.values())), **counts}
    flat_dist = [d for _, d in eligible_services]
    tobler_unit = [tobler_unit_map.get(s['nearest_node']) for s, _ in eligible_services]
    tobler_unit = [float(x) for x in tobler_unit if x is not None and np.isfinite(x)]
    for v in WALKING_SPEEDS_MS:
        tag = str(v).replace('.', 'p')
        flat_times = [d / v for d in flat_dist]
        tobler_times = [c / v for c in tobler_unit]
        result[f'tempo_medio_seg__flat_{tag}'] = float(np.mean(flat_times)) if flat_times else np.nan
        result[f'tempo_min_seg__flat_{tag}'] = float(np.min(flat_times)) if flat_times else np.nan
        result[f'tempo_medio_seg__tobler_{tag}'] = float(np.mean(tobler_times)) if tobler_times else np.nan
        result[f'tempo_min_seg__tobler_{tag}'] = float(np.min(tobler_times)) if tobler_times else np.nan
    result['tempo_medio_servicos_seg'] = result['tempo_medio_seg__tobler_0p7']
    result['tempo_minimo_servico_seg'] = result['tempo_min_seg__tobler_0p7']
    result['cenario_principal'] = PRIMARY_SCENARIO
    result['distancia_max_m'] = max_distance
    result['time_hard_threshold'] = False
    return (result, od_records)

def process_subarea(subarea_geometry, output_name):
    nome_lower = output_name.lower()
    if 'northwest' in nome_lower:
        quadrante = 'northwest'
    elif 'northeast' in nome_lower:
        quadrante = 'northeast'
    elif 'southwest' in nome_lower:
        quadrante = 'southwest'
    elif 'southeast' in nome_lower:
        quadrante = 'southeast'
    else:
        raise ValueError(f'Workflow validation diagnostic.{output_name}')
    subarea_buildings = connected_buildings_global[connected_buildings_global['_quadrante'] == quadrante].copy()
    if TEST_MODE and len(subarea_buildings) > TEST_N_BUILDINGS_PER_SUBAREA:
        subarea_buildings = subarea_buildings.sample(TEST_N_BUILDINGS_PER_SUBAREA, random_state=42).copy()
        print(f'TEST_MODE: {output_name} limitado a {len(subarea_buildings)}Building-universe validation failed.')
    valid_services = services_buffer_800m[services_buffer_800m[category_col].isin(target_categories)].copy()
    subarea_buildings['nearest_node'] = subarea_buildings['geometry'].apply(lambda geom: nearest_graph_node(geom, node_tree, node_list))
    valid_services['nearest_node'] = valid_services['geometry'].apply(lambda geom: nearest_graph_node(geom, node_tree, node_list))
    if subarea_buildings['nearest_node'].isna().any():
        raise ValueError(f'{output_name}Building-universe validation failed.')
    if valid_services['nearest_node'].isna().any():
        raise ValueError(f'{output_name}Service-data validation failed.')
    id_col = 'osm_id' if 'osm_id' in subarea_buildings.columns else None
    if id_col is None:
        subarea_buildings['_building_id_tmp'] = subarea_buildings.index.astype(str)
        id_col = '_building_id_tmp'
    print(f'{output_name}Building-universe validation failed.{len(subarea_buildings)}Service-data validation failed.{len(valid_services)}')
    pairs = list(zip(subarea_buildings[id_col].astype(str), subarea_buildings['nearest_node']))
    calc = Parallel(n_jobs=-1, prefer='processes')((delayed(calculate_building_metrics)(bid, node, valid_services, MAX_NETWORK_DISTANCE_M) for bid, node in pairs))
    results = [x[0] for x in calc]
    od_lists = [x[1] for x in calc]
    results_df = pd.DataFrame(results, index=subarea_buildings.index)
    building_results = subarea_buildings.join(results_df)
    assert len(building_results) == len(subarea_buildings)
    assert (pd.to_numeric(building_results['numero_servicos_proximos'], errors='coerce').fillna(0) >= 0).all()
    if SAVE_OD_LONG:
        od_records = [row for rows in od_lists for row in rows]
        od_df = pd.DataFrame(od_records)
        od_path = f'{output_name}_od800.csv'
        od_df.to_csv(od_path, index=False)
        print(f'OD guardada: {od_path} | pairs edifício-serviço: {len(od_df)}')
    print(f'{output_name}Building-universe validation failed.{len(building_results)}')
    return building_results

## Topology quality control

Each computational subarea follows the same sequence:

1. calculate the base accessibility output;
2. identify implausible network-versus-straight-line cases;
3. group suspect cases into local hotspots;
4. evaluate candidate network links;
5. accept only links that satisfy the quantitative improvement criteria;
6. assign raster-directional slope to accepted links;
7. recalculate the affected subarea;
8. save the corrected subarea output.

This step is a reproducible network-quality-control procedure and does not alter the service inventory or the 800 m accessibility specification.


In [ ]:
import ast
import itertools
import math
from pathlib import Path
import numpy as np
import pandas as pd
import geopandas as gpd
import networkx as nx
from shapely import wkt
from shapely.geometry import Point, LineString
GRAPH_CRS = 'EPSG:3763'
SUSPECT_RATIO_THRESHOLD = 1.5
SUSPECT_EXCESS_THRESHOLD = 50.0
CORRIDOR_BUFFER_M = 35
MAX_LINK_DIST_M = 20
TOP_K_CANDIDATES = 20
MIN_CASES_IMPROVED = 5
MIN_MEAN_IMPROVEMENT = 30.0
MAX_ITERS_PER_HOTSPOT = 5
HOTSPOT_GRID_SIZE_M = 150
MIN_CASES_PER_HOTSPOT = 5
SERVICES_CSV = str(SERVICES_FILE)

def safe_wkt_load(v):
    if pd.isna(v):
        return None
    s = str(v).strip()
    if not s:
        return None
    try:
        return wkt.loads(s)
    except Exception:
        return None

def safe_tuple_to_point(v):
    if pd.isna(v):
        return None
    s = str(v).strip()
    if not s:
        return None
    try:
        t = ast.literal_eval(s)
        if isinstance(t, (tuple, list)) and len(t) >= 2:
            x, y = (float(t[0]), float(t[1]))
            if np.isfinite(x) and np.isfinite(y):
                return Point(x, y)
    except Exception:
        return None
    return None

def valid_geom(g):
    if g is None or not hasattr(g, 'is_empty') or g.is_empty:
        return False
    try:
        return all((np.isfinite(v) for v in g.bounds))
    except Exception:
        return False

def geom_to_point(g):
    if not valid_geom(g):
        return None
    if g.geom_type == 'Point':
        return g
    try:
        rp = g.representative_point()
        if valid_geom(rp):
            return rp
    except Exception:
        pass
    return None

def get_node_xy(node, data=None):
    if data is None:
        data = {}
    if isinstance(data, dict):
        if 'x' in data and 'y' in data:
            return (float(data['x']), float(data['y']))
        if 'lon' in data and 'lat' in data:
            return (float(data['lon']), float(data['lat']))
        if 'X' in data and 'Y' in data:
            return (float(data['X']), float(data['Y']))
        if 'geometry' in data and data['geometry'] is not None:
            geom = data['geometry']
            if hasattr(geom, 'geom_type') and geom.geom_type == 'Point' and (not geom.is_empty):
                return (float(geom.x), float(geom.y))
    if isinstance(node, (tuple, list)) and len(node) >= 2:
        try:
            return (float(node[0]), float(node[1]))
        except Exception:
            pass
    return None

def nearest_graph_node_to_point(G, point_xy):
    px, py = point_xy
    best_node = None
    best_d2 = None
    for n, data in G.nodes(data=True):
        xy = get_node_xy(n, data)
        if xy is None:
            continue
        nx_, ny_ = xy
        d2 = (nx_ - px) ** 2 + (ny_ - py) ** 2
        if best_d2 is None or d2 < best_d2:
            best_d2 = d2
            best_node = n
    return best_node

def shortest_path_nodes(G, source, target):
    try:
        return nx.shortest_path(G, source=source, target=target, weight='length')
    except Exception:
        return nx.shortest_path(G, source=source, target=target)

def path_to_linestring(G, path_nodes):
    coords = []
    for n in path_nodes:
        xy = get_node_xy(n, G.nodes[n])
        if xy is None:
            continue
        coords.append(xy)
    if len(coords) < 2:
        return None
    try:
        return LineString(coords)
    except Exception:
        return None

def route_length_between_nodes(G, source, target):
    try:
        path_nodes = shortest_path_nodes(G, source, target)
        path_line = path_to_linestring(G, path_nodes)
        if path_line is None or path_line.is_empty:
            return (float('nan'), None, None)
        return (float(path_line.length), path_nodes, path_line)
    except Exception:
        return (float('nan'), None, None)

def load_services_gdf(services_csv=SERVICES_CSV):
    serv = pd.read_csv(services_csv)
    serv = serv.dropna(subset=['Longitude', 'Latitude']).copy()
    serv['Name_norm'] = serv['Name'].astype(str).str.strip().str.lower()
    return gpd.GeoDataFrame(serv, geometry=gpd.points_from_xy(serv['Longitude'], serv['Latitude']), crs='EPSG:4326').to_crs(GRAPH_CRS)

def load_area_results_for_diagnostics(area_csv, services_gdf):
    df = pd.read_csv(area_csv).copy()
    df['_nearest_node_pt'] = df['nearest_node'].apply(safe_tuple_to_point)
    df['_geom_edificio'] = df['geometry_wkt'].apply(safe_wkt_load)

    def choose_origin(row):
        if valid_geom(row['_nearest_node_pt']):
            return row['_nearest_node_pt']
        return geom_to_point(row['_geom_edificio'])
    df['_origem_atual'] = df.apply(choose_origin, axis=1)
    df['servico_min_id_norm'] = df['servico_min_id'].astype(str).str.strip().str.lower()
    serv_join = services_gdf[['Name_norm', 'Category', 'geometry']].copy()
    serv_join = serv_join.rename(columns={'geometry': '_destino_geom', 'Category': '_destino_categoria_real'})
    merged = df.merge(serv_join, left_on='servico_min_id_norm', right_on='Name_norm', how='left')
    if len(merged) > len(df):
        merged = gpd.GeoDataFrame(merged, geometry='_destino_geom', crs=GRAPH_CRS)
        merged['_dist_match'] = merged.apply(lambda r: r['_origem_atual'].distance(r['_destino_geom']) if valid_geom(r['_origem_atual']) and valid_geom(r['_destino_geom']) else np.nan, axis=1)
        merged = merged.sort_values(['osm_id', '_dist_match'])
        merged = merged.groupby('osm_id', as_index=False).first()
    merged = gpd.GeoDataFrame(merged, geometry='_destino_geom', crs=GRAPH_CRS)
    merged['_dist_direta_m'] = merged.apply(lambda r: r['_origem_atual'].distance(r['_destino_geom']) if valid_geom(r['_origem_atual']) and valid_geom(r['_destino_geom']) else np.nan, axis=1)
    merged['_dist_rede_m'] = pd.to_numeric(merged['distancia_minima_servico'], errors='coerce')
    merged['_excesso_abs_m'] = merged['_dist_rede_m'] - merged['_dist_direta_m']
    merged['_ratio_rede_direta'] = merged['_dist_rede_m'] / merged['_dist_direta_m']
    merged['_flag_suspeito'] = merged['_dist_direta_m'].notna() & (merged['_dist_direta_m'] > 0) & (merged['_ratio_rede_direta'] > SUSPECT_RATIO_THRESHOLD) & (merged['_excesso_abs_m'] > SUSPECT_EXCESS_THRESHOLD)
    return merged

def hotspot_grid_labels(cases_df, cell_size_m=HOTSPOT_GRID_SIZE_M):
    tmp = cases_df.copy()
    tmp['_x'] = tmp['_origem_atual'].apply(lambda g: g.x if valid_geom(g) else np.nan)
    tmp['_y'] = tmp['_origem_atual'].apply(lambda g: g.y if valid_geom(g) else np.nan)
    tmp['_gx'] = np.floor(tmp['_x'] / cell_size_m).astype('Int64')
    tmp['_gy'] = np.floor(tmp['_y'] / cell_size_m).astype('Int64')
    tmp['hotspot_id'] = tmp['_gx'].astype(str) + '_' + tmp['_gy'].astype(str)
    return tmp

def detect_hotspots(area_diag_df, min_cases_per_hotspot=MIN_CASES_PER_HOTSPOT):
    suspects = area_diag_df[area_diag_df['_flag_suspeito']].copy()
    suspects = suspects[suspects['_origem_atual'].notna()].copy()
    if len(suspects) == 0:
        return (suspects, pd.DataFrame(columns=['hotspot_id', 'n_cases']))
    suspects = hotspot_grid_labels(suspects, HOTSPOT_GRID_SIZE_M)
    counts = suspects.groupby('hotspot_id', as_index=False).size().rename(columns={'size': 'n_cases'}).sort_values('n_cases', ascending=False)
    keep = counts[counts['n_cases'] >= min_cases_per_hotspot]['hotspot_id'].tolist()
    suspects = suspects[suspects['hotspot_id'].isin(keep)].copy()
    counts = counts[counts['hotspot_id'].isin(keep)].copy()
    return (suspects, counts)

def point_in_polygon(pt, polygon_geom):
    if not valid_geom(pt):
        return False
    return pt.within(polygon_geom) or pt.intersects(polygon_geom)

def build_case_direct_line(row):
    origin = row['_origem_atual']
    destination = row['_destino_geom']
    if valid_geom(origin) and valid_geom(destination):
        try:
            return LineString([origin, destination])
        except Exception:
            return None
    return None

def build_corridor_from_cases(cases_df, buffer_m=CORRIDOR_BUFFER_M):
    geoms = []
    for _, row in cases_df.iterrows():
        line = build_case_direct_line(row)
        if line is not None and (not line.is_empty):
            geoms.append(line.buffer(buffer_m))
    if not geoms:
        return None
    return gpd.GeoSeries(geoms, crs=GRAPH_CRS).union_all()

def extract_subgraph_nodes_in_geom(G, polygon_geom):
    inside_nodes = []
    for n, data in G.nodes(data=True):
        xy = get_node_xy(n, data)
        if xy is None:
            continue
        p = Point(xy)
        if p.within(polygon_geom) or p.intersects(polygon_geom):
            inside_nodes.append(n)
    return inside_nodes

def make_nodes_gdf(G_sub):
    rows = []
    for n, data in G_sub.nodes(data=True):
        xy = get_node_xy(n, data)
        if xy is None:
            continue
        rows.append({'node': n, 'degree': G_sub.degree(n), 'geometry': Point(xy)})
    return gpd.GeoDataFrame(rows, geometry='geometry', crs=GRAPH_CRS)

def find_endpoint_candidates_in_corridor(G, cases_df):
    corridor = build_corridor_from_cases(cases_df, CORRIDOR_BUFFER_M)
    if corridor is None or corridor.is_empty:
        return gpd.GeoDataFrame(columns=['node_a', 'node_b', 'dist_m', 'score', 'geometry'], geometry='geometry', crs=GRAPH_CRS)
    sub_nodes = extract_subgraph_nodes_in_geom(G, corridor)
    G_sub = G.subgraph(sub_nodes).copy()
    nodes_gdf = make_nodes_gdf(G_sub)
    endpoints = nodes_gdf[nodes_gdf['degree'] == 1].copy()
    rows = []
    recs = endpoints.to_dict('records')
    for a, b in itertools.combinations(recs, 2):
        pa = a['geometry']
        pb = b['geometry']
        d = pa.distance(pb)
        if d == 0 or d > MAX_LINK_DIST_M:
            continue
        na = a['node']
        nb = b['node']
        if G_sub.has_edge(na, nb) or G_sub.has_edge(nb, na):
            continue
        seg = LineString([pa, pb])
        score = seg.distance(corridor) + d
        rows.append({'node_a': na, 'node_b': nb, 'dist_m': d, 'score': score, 'geometry': seg})
    cand = gpd.GeoDataFrame(rows, geometry='geometry', crs=GRAPH_CRS)
    if len(cand) == 0:
        return cand
    return cand.sort_values(['score', 'dist_m'], ascending=[True, True]).reset_index(drop=True)

def evaluate_candidate_on_cases(G_base, cases_df, node_a, node_b, link_dist_m):
    G_tmp = G_base.copy()
    G_tmp.add_edge(node_a, node_b, length=link_dist_m)
    G_tmp.add_edge(node_b, node_a, length=link_dist_m)
    rows = []
    for _, row in cases_df.iterrows():
        origin = row['_origem_atual']
        destination = row['_destino_geom']
        if not valid_geom(origin) or not valid_geom(destination):
            continue
        origem_node_before = nearest_graph_node_to_point(G_base, (origin.x, origin.y))
        destino_node_before = nearest_graph_node_to_point(G_base, (destination.x, destination.y))
        origem_node_after = nearest_graph_node_to_point(G_tmp, (origin.x, origin.y))
        destino_node_after = nearest_graph_node_to_point(G_tmp, (destination.x, destination.y))
        dist_before, _, _ = route_length_between_nodes(G_base, origem_node_before, destino_node_before)
        dist_after, _, _ = route_length_between_nodes(G_tmp, origem_node_after, destino_node_after)
        melhoria = dist_before - dist_after if pd.notna(dist_before) and pd.notna(dist_after) else np.nan
        rows.append({'osm_id': row['osm_id'], 'dist_before': dist_before, 'dist_after': dist_after, 'melhoria_m': melhoria})
    eval_df = pd.DataFrame(rows)
    n_improved = int((eval_df['melhoria_m'] > 0).sum())
    mean_improvement = float(eval_df.loc[eval_df['melhoria_m'] > 0, 'melhoria_m'].mean()) if n_improved > 0 else 0.0
    median_improvement = float(eval_df.loc[eval_df['melhoria_m'] > 0, 'melhoria_m'].median()) if n_improved > 0 else 0.0
    total_improvement = float(eval_df['melhoria_m'].clip(lower=0).sum()) if len(eval_df) else 0.0
    return {'node_a': node_a, 'node_b': node_b, 'link_dist_m': link_dist_m, 'n_cases': len(eval_df), 'n_improved': n_improved, 'mean_improvement_m': mean_improvement, 'median_improvement_m': median_improvement, 'total_improvement_m': total_improvement, 'eval_df': eval_df, 'G_tmp': G_tmp}

def search_best_candidate_for_hotspot(G_base, cases_df):
    cand = find_endpoint_candidates_in_corridor(G_base, cases_df)
    if len(cand) == 0:
        return (None, cand, pd.DataFrame())
    tested = []
    for _, c in cand.head(TOP_K_CANDIDATES).iterrows():
        result = evaluate_candidate_on_cases(G_base, cases_df, c['node_a'], c['node_b'], c['dist_m'])
        tested.append({'node_a': result['node_a'], 'node_b': result['node_b'], 'link_dist_m': result['link_dist_m'], 'n_cases': result['n_cases'], 'n_improved': result['n_improved'], 'mean_improvement_m': result['mean_improvement_m'], 'median_improvement_m': result['median_improvement_m'], 'total_improvement_m': result['total_improvement_m']})
    tested_df = pd.DataFrame(tested)
    tested_df = tested_df.sort_values(['n_improved', 'total_improvement_m', 'mean_improvement_m'], ascending=[False, False, False]).reset_index(drop=True)
    best = tested_df.iloc[0].to_dict() if len(tested_df) else None
    return (best, cand, tested_df)

def iterative_hotspot_fix(G_start, cases_df):
    G_current = G_start.copy()
    accepted_links = []
    remaining_cases = cases_df.copy()
    for it in range(1, MAX_ITERS_PER_HOTSPOT + 1):
        best, cand_df, tested_df = search_best_candidate_for_hotspot(G_current, remaining_cases)
        print(f'\n=== Iteração {it} ===')
        print('Workflow validation diagnostic.', len(remaining_cases))
        if best is None:
            print('Sem candidatas úteis.')
            break
        display(tested_df.head(10))
        if best['n_improved'] < MIN_CASES_IMPROVED or best['mean_improvement_m'] < MIN_MEAN_IMPROVEMENT:
            print('Workflow validation diagnostic.')
            break
        print('Aceite:')
        print(best)
        G_current.add_edge(best['node_a'], best['node_b'], length=best['link_dist_m'])
        G_current.add_edge(best['node_b'], best['node_a'], length=best['link_dist_m'])
        accepted_links.append({'iter': it, **best})
        result = evaluate_candidate_on_cases(G_start if it == 1 else G_current, remaining_cases, best['node_a'], best['node_b'], best['link_dist_m'])
        eval_df = result['eval_df'][['osm_id', 'melhoria_m']].copy()
        remaining_cases = remaining_cases.merge(eval_df, on='osm_id', how='left')
        remaining_cases = remaining_cases[remaining_cases['melhoria_m'].fillna(0) <= 0].copy()
        remaining_cases = remaining_cases.drop(columns=['melhoria_m'])
        if len(remaining_cases) == 0:
            print('Topology-quality-control diagnostic.')
            break
    accepted_df = pd.DataFrame(accepted_links)
    return (G_current, accepted_df, remaining_cases)

def set_routing_graph_context(G_graph):
    """Atualiza o contexto global usado por processar_subarea sem alterar a função original."""
    global G_main, node_list, node_tree
    G_main = G_graph.copy()
    node_list = list(G_main.nodes)
    node_tree = cKDTree(np.asarray(node_list, dtype=float))
    return G_main

def process_subarea_with_graph(subarea_geometry, output_name, G_graph):
    """Workflow validation diagnostic."""
    global G_main, node_list, node_tree
    old_G_main = G_main
    old_nos_lista = node_list
    old_arvore_nos = node_tree
    try:
        set_routing_graph_context(G_graph)
        return process_subarea(subarea_geometry, output_name)
    finally:
        G_main = old_G_main
        node_list = old_nos_lista
        node_tree = old_arvore_nos

def save_area_result_csv(gdf_result, csv_path):
    out = gdf_result.copy()
    out['geometry_wkt'] = out.geometry.to_wkt()
    out.drop(columns='geometry').to_csv(csv_path, index=False)

def print_area_correction_report(area_name, area_diag_df, suspects_df, hotspot_counts_df, accepted_links_df, final_result_gdf=None):
    print('\n' + '=' * 80)
    print(f'RELATÓRIO DA ÁREA: {area_name.upper()}')
    print('=' * 80)
    total_registos = len(area_diag_df) if area_diag_df is not None else 0
    total_suspects = len(suspects_df) if suspects_df is not None else 0
    perc = 100.0 * total_suspects / total_registos if total_registos else 0.0
    print('Resumo base da área')
    print('- Total registos:', total_registos)
    print('Workflow validation diagnostic.', total_suspects)
    print('- Percentagem suspect cases:', round(perc, 2), '%')
    if area_diag_df is not None and len(area_diag_df) > 0:
        dist_media = pd.to_numeric(area_diag_df.get('_dist_rede_m'), errors='coerce').mean()
        ratio_med = pd.to_numeric(area_diag_df.get('_ratio_rede_direta'), errors='coerce').median()
        print('Workflow validation diagnostic.', round(dist_media, 3) if pd.notna(dist_media) else None)
        print('Workflow validation diagnostic.', round(ratio_med, 3) if pd.notna(ratio_med) else None)
    print('\nHotspots')
    if hotspot_counts_df is not None and len(hotspot_counts_df) > 0:
        print('- Nº hotspots:', len(hotspot_counts_df))
        display(hotspot_counts_df.head(10))
    else:
        print('- No hotspot found.')
    print('\nAccepted links')
    if accepted_links_df is not None and len(accepted_links_df) > 0:
        print('- Nº accepted links:', len(accepted_links_df))
        print('Workflow validation diagnostic.', int(pd.to_numeric(accepted_links_df.get('n_improved'), errors='coerce').fillna(0).sum()))
        mean_imp = pd.to_numeric(accepted_links_df.get('mean_improvement_m'), errors='coerce').mean()
        total_imp = pd.to_numeric(accepted_links_df.get('total_improvement_m'), errors='coerce').sum()
        print('Workflow validation diagnostic.', round(mean_imp, 3) if pd.notna(mean_imp) else None)
        print('Workflow validation diagnostic.', round(total_imp, 3) if pd.notna(total_imp) else None)
        display(accepted_links_df)
    else:
        print('Workflow validation diagnostic.')
    if final_result_gdf is not None:
        print('Workflow validation diagnostic.')
        print('- Registos finais:', len(final_result_gdf))
        if 'distancia_minima_servico' in final_result_gdf.columns:
            d = pd.to_numeric(final_result_gdf['distancia_minima_servico'], errors='coerce')
            print('Workflow validation diagnostic.', round(d.mean(), 3) if pd.notna(d.mean()) else None)
            print('Workflow validation diagnostic.', round(d.median(), 3) if pd.notna(d.median()) else None)

def run_inline_topology_correction(area_name, subarea_geometry, output_name, base_csv, final_csv=None, services_csv=SERVICES_CSV):
    """Workflow validation diagnostic."""
    if final_csv is None:
        final_csv = base_csv
    G_area_base = G_main.copy()
    print(f'Workflow validation diagnostic.{area_name}')
    print('CSV base:', base_csv)
    print('CSV final:', final_csv)
    G_corr, accepted_links_df, area_diag_df, suspects_df, hotspot_counts_df = run_topology_correction_for_area(area_name=area_name, area_csv=base_csv, G_area=G_area_base, services_csv=services_csv)
    print_area_correction_report(area_name=area_name, area_diag_df=area_diag_df, suspects_df=suspects_df, hotspot_counts_df=hotspot_counts_df, accepted_links_df=accepted_links_df, final_result_gdf=None)
    print(f'Workflow validation diagnostic.{area_name}Workflow validation diagnostic.')
    corrected_result = process_subarea_with_graph(subarea_geometry=subarea_geometry, output_name=f'{output_name}_corr', G_graph=G_corr)
    print_area_correction_report(area_name=f'{area_name} (após recálculo)', area_diag_df=area_diag_df, suspects_df=suspects_df, hotspot_counts_df=hotspot_counts_df, accepted_links_df=accepted_links_df, final_result_gdf=corrected_result)
    print(f'\n>>> Guardar CSV final corrigido da área {area_name}...')
    save_area_result_csv(corrected_result, final_csv)
    print('Final CSV saved to:', final_csv)
    return {'resultado_corr': corrected_result, 'G_corr': G_corr, 'accepted_links_df': accepted_links_df, 'area_diag_df': area_diag_df, 'suspects_df': suspects_df, 'hotspot_counts_df': hotspot_counts_df, 'base_csv': base_csv, 'final_csv': final_csv}

## 6. Process each computational subarea and apply topology quality control

In [ ]:
edificios_northwest_lordelo_ouro_massarelos_resultado = process_subarea(northwest_geom, 'northwest_lordelo_ouro_massarelos')
save_area_result_csv(edificios_northwest_lordelo_ouro_massarelos_resultado, 'lordelo_ouro_massarelos_northwest_base.csv')

In [ ]:
def point_in_polygon(pt, polygon_geom):
    if not valid_geom(pt):
        return False
    return pt.within(polygon_geom) or pt.intersects(polygon_geom)

def build_case_direct_line(row):
    origin = row['_origem_atual']
    destination = row['_destino_geom']
    if valid_geom(origin) and valid_geom(destination):
        try:
            return LineString([origin, destination])
        except Exception:
            return None
    return None

def build_corridor_from_cases(cases_df, buffer_m=CORRIDOR_BUFFER_M):
    geoms = []
    for _, row in cases_df.iterrows():
        line = build_case_direct_line(row)
        if line is not None and (not line.is_empty):
            geoms.append(line.buffer(buffer_m))
    if not geoms:
        return None
    return gpd.GeoSeries(geoms, crs=GRAPH_CRS).union_all()

def extract_subgraph_nodes_in_geom(G, polygon_geom):
    inside_nodes = []
    for n, data in G.nodes(data=True):
        xy = get_node_xy(n, data)
        if xy is None:
            continue
        p = Point(xy)
        if p.within(polygon_geom) or p.intersects(polygon_geom):
            inside_nodes.append(n)
    return inside_nodes

def make_nodes_gdf(G_sub):
    rows = []
    for n, data in G_sub.nodes(data=True):
        xy = get_node_xy(n, data)
        if xy is None:
            continue
        rows.append({'node': n, 'degree': G_sub.degree(n), 'geometry': Point(xy)})
    if len(rows) == 0:
        return gpd.GeoDataFrame(columns=['node', 'degree', 'geometry'], geometry='geometry', crs=GRAPH_CRS)
    return gpd.GeoDataFrame(rows, geometry='geometry', crs=GRAPH_CRS)

def find_endpoint_candidates_in_corridor(G, cases_df):
    corridor = build_corridor_from_cases(cases_df, CORRIDOR_BUFFER_M)
    if corridor is None or corridor.is_empty:
        return gpd.GeoDataFrame(columns=['node_a', 'node_b', 'dist_m', 'score', 'geometry'], geometry='geometry', crs=GRAPH_CRS)
    sub_nodes = extract_subgraph_nodes_in_geom(G, corridor)
    G_sub = G.subgraph(sub_nodes).copy()
    nodes_gdf = make_nodes_gdf(G_sub)
    if len(nodes_gdf) == 0:
        return gpd.GeoDataFrame(columns=['node_a', 'node_b', 'dist_m', 'score', 'geometry'], geometry='geometry', crs=GRAPH_CRS)
    endpoints = nodes_gdf[nodes_gdf['degree'] == 1].copy()
    if len(endpoints) < 2:
        return gpd.GeoDataFrame(columns=['node_a', 'node_b', 'dist_m', 'score', 'geometry'], geometry='geometry', crs=GRAPH_CRS)
    rows = []
    recs = endpoints.to_dict('records')
    for a, b in itertools.combinations(recs, 2):
        pa = a['geometry']
        pb = b['geometry']
        d = pa.distance(pb)
        if d == 0 or d > MAX_LINK_DIST_M:
            continue
        na = a['node']
        nb = b['node']
        if G_sub.has_edge(na, nb) or G_sub.has_edge(nb, na):
            continue
        seg = LineString([pa, pb])
        score = seg.distance(corridor) + d
        rows.append({'node_a': na, 'node_b': nb, 'dist_m': d, 'score': score, 'geometry': seg})
    if len(rows) == 0:
        return gpd.GeoDataFrame(columns=['node_a', 'node_b', 'dist_m', 'score', 'geometry'], geometry='geometry', crs=GRAPH_CRS)
    if rows:
        cand = gpd.GeoDataFrame(rows, geometry='geometry', crs=GRAPH_CRS)
    else:
        cand = gpd.GeoDataFrame(columns=['node_a', 'node_b', 'distance_m', 'score', 'geometry'], geometry='geometry', crs=GRAPH_CRS)
    return cand.sort_values(['score', 'dist_m'], ascending=[True, True]).reset_index(drop=True)

def evaluate_candidate_on_cases(G_base, cases_df, node_a, node_b, link_dist_m):
    G_tmp = G_base.copy()
    G_tmp.add_edge(node_a, node_b, length=link_dist_m)
    G_tmp.add_edge(node_b, node_a, length=link_dist_m)
    rows = []
    for _, row in cases_df.iterrows():
        origin = row['_origem_atual']
        destination = row['_destino_geom']
        if not valid_geom(origin) or not valid_geom(destination):
            continue
        origem_node_before = nearest_graph_node_to_point(G_base, (origin.x, origin.y))
        destino_node_before = nearest_graph_node_to_point(G_base, (destination.x, destination.y))
        origem_node_after = nearest_graph_node_to_point(G_tmp, (origin.x, origin.y))
        destino_node_after = nearest_graph_node_to_point(G_tmp, (destination.x, destination.y))
        dist_before, _, _ = route_length_between_nodes(G_base, origem_node_before, destino_node_before)
        dist_after, _, _ = route_length_between_nodes(G_tmp, origem_node_after, destino_node_after)
        melhoria = dist_before - dist_after if pd.notna(dist_before) and pd.notna(dist_after) else np.nan
        rows.append({'osm_id': row['osm_id'], 'dist_before': dist_before, 'dist_after': dist_after, 'melhoria_m': melhoria})
    eval_df = pd.DataFrame(rows)
    n_improved = int((eval_df['melhoria_m'] > 0).sum()) if len(eval_df) else 0
    mean_improvement = float(eval_df.loc[eval_df['melhoria_m'] > 0, 'melhoria_m'].mean()) if n_improved > 0 else 0.0
    median_improvement = float(eval_df.loc[eval_df['melhoria_m'] > 0, 'melhoria_m'].median()) if n_improved > 0 else 0.0
    total_improvement = float(eval_df['melhoria_m'].clip(lower=0).sum()) if len(eval_df) else 0.0
    return {'node_a': node_a, 'node_b': node_b, 'link_dist_m': link_dist_m, 'n_cases': len(eval_df), 'n_improved': n_improved, 'mean_improvement_m': mean_improvement, 'median_improvement_m': median_improvement, 'total_improvement_m': total_improvement, 'eval_df': eval_df, 'G_tmp': G_tmp}

def search_best_candidate_for_hotspot(G_base, cases_df):
    cand = find_endpoint_candidates_in_corridor(G_base, cases_df)
    if cand is None or len(cand) == 0:
        return (None, cand, pd.DataFrame(columns=['node_a', 'node_b', 'link_dist_m', 'n_cases', 'n_improved', 'mean_improvement_m', 'median_improvement_m', 'total_improvement_m']))
    tested = []
    for _, c in cand.head(TOP_K_CANDIDATES).iterrows():
        result = evaluate_candidate_on_cases(G_base, cases_df, c['node_a'], c['node_b'], c['dist_m'])
        tested.append({'node_a': result['node_a'], 'node_b': result['node_b'], 'link_dist_m': result['link_dist_m'], 'n_cases': result['n_cases'], 'n_improved': result['n_improved'], 'mean_improvement_m': result['mean_improvement_m'], 'median_improvement_m': result['median_improvement_m'], 'total_improvement_m': result['total_improvement_m']})
    if len(tested) == 0:
        return (None, cand, pd.DataFrame(columns=['node_a', 'node_b', 'link_dist_m', 'n_cases', 'n_improved', 'mean_improvement_m', 'median_improvement_m', 'total_improvement_m']))
    tested_df = pd.DataFrame(tested)
    tested_df = tested_df.sort_values(['n_improved', 'total_improvement_m', 'mean_improvement_m'], ascending=[False, False, False]).reset_index(drop=True)
    best = tested_df.iloc[0].to_dict()
    return (best, cand, tested_df)

def iterative_hotspot_fix(G_start, cases_df):
    G_current = G_start.copy()
    accepted_links = []
    remaining_cases = cases_df.copy()
    for it in range(1, MAX_ITERS_PER_HOTSPOT + 1):
        best, cand_df, tested_df = search_best_candidate_for_hotspot(G_current, remaining_cases)
        print(f'\n=== Iteração {it} ===')
        print('Workflow validation diagnostic.', len(remaining_cases))
        if best is None:
            print('Sem candidatas úteis.')
            break
        display(tested_df.head(10))
        if best['n_improved'] < MIN_CASES_IMPROVED or best['mean_improvement_m'] < MIN_MEAN_IMPROVEMENT:
            print('Workflow validation diagnostic.')
            break
        print('Aceite:')
        print(best)
        G_current.add_edge(best['node_a'], best['node_b'], length=best['link_dist_m'])
        G_current.add_edge(best['node_b'], best['node_a'], length=best['link_dist_m'])
        accepted_links.append({'iter': it, **best})
        result = evaluate_candidate_on_cases(G_start if it == 1 else G_current, remaining_cases, best['node_a'], best['node_b'], best['link_dist_m'])
        eval_df = result['eval_df'][['osm_id', 'melhoria_m']].copy()
        remaining_cases = remaining_cases.merge(eval_df, on='osm_id', how='left')
        remaining_cases = remaining_cases[remaining_cases['melhoria_m'].fillna(0) <= 0].copy()
        remaining_cases = remaining_cases.drop(columns=['melhoria_m'])
        if len(remaining_cases) == 0:
            print('Topology-quality-control diagnostic.')
            break
    accepted_df = pd.DataFrame(accepted_links)
    return (G_current, accepted_df, remaining_cases)

In [ ]:
def validate_service_selection(result_gdf, services_csv=SERVICES_CSV, graph_crs=GRAPH_CRS, ratio_threshold=2.5, excess_threshold=150.0, max_direct_fallback_m=80.0, replace_invalid=False):
    """Service-data validation failed."""
    out = result_gdf.copy()

    def safe_parse_nearest_node(v):
        if pd.isna(v):
            return None
        s = str(v).strip()
        if not s:
            return None
        try:
            t = ast.literal_eval(s)
            if isinstance(t, (tuple, list)) and len(t) >= 2:
                x, y = (float(t[0]), float(t[1]))
                if np.isfinite(x) and np.isfinite(y):
                    return Point(x, y)
        except Exception:
            pass
        return None
    if 'nearest_node' in out.columns:
        out['_origem_pt'] = out['nearest_node'].apply(safe_parse_nearest_node)
    else:
        out['_origem_pt'] = None
    if '_origem_pt' not in out.columns or out['_origem_pt'].isna().all():
        if 'geometry' in out.columns:
            out['_origem_pt'] = out.geometry.apply(geom_to_point)
        else:
            out['_origem_pt'] = None
    serv = pd.read_csv(services_csv)
    serv = serv.dropna(subset=['Longitude', 'Latitude']).copy()
    serv['Name_norm'] = serv['Name'].astype(str).str.strip().str.lower()
    serv_gdf = gpd.GeoDataFrame(serv, geometry=gpd.points_from_xy(serv['Longitude'], serv['Latitude']), crs='EPSG:4326').to_crs(graph_crs)
    serv_join = serv_gdf[['Name_norm', 'Name', 'Category', 'geometry']].copy()
    serv_join = serv_join.rename(columns={'geometry': '_destino_geom', 'Name': '_destino_nome_real', 'Category': '_destino_categoria_real'})
    out['servico_min_id_norm'] = out['servico_min_id'].astype(str).str.strip().str.lower()
    merged = out.merge(serv_join, left_on='servico_min_id_norm', right_on='Name_norm', how='left')
    if len(merged) > len(out):
        merged = gpd.GeoDataFrame(merged, geometry='geometry', crs=graph_crs)
        merged['_dist_match'] = merged.apply(lambda r: r['_origem_pt'].distance(r['_destino_geom']) if valid_geom(r['_origem_pt']) and valid_geom(r['_destino_geom']) else np.nan, axis=1)
        merged = merged.sort_values(['osm_id', '_dist_match'])
        merged = merged.groupby('osm_id', as_index=False).first()
    merged = gpd.GeoDataFrame(merged, geometry='geometry', crs=result_gdf.crs)
    merged['_dist_direta_servico_escolhido_m'] = merged.apply(lambda r: r['_origem_pt'].distance(r['_destino_geom']) if valid_geom(r['_origem_pt']) and valid_geom(r['_destino_geom']) else np.nan, axis=1)
    merged['_dist_rede_servico_escolhido_m'] = pd.to_numeric(merged['distancia_minima_servico'], errors='coerce')
    merged['_ratio_rede_direta'] = merged['_dist_rede_servico_escolhido_m'] / merged['_dist_direta_servico_escolhido_m']
    merged['_excesso_rede_vs_direta_m'] = merged['_dist_rede_servico_escolhido_m'] - merged['_dist_direta_servico_escolhido_m']
    merged['ligacao_servico_valida'] = ~(merged['_dist_direta_servico_escolhido_m'].notna() & (merged['_dist_direta_servico_escolhido_m'] > 0) & ((merged['_ratio_rede_direta'] > ratio_threshold) & (merged['_excesso_rede_vs_direta_m'] > excess_threshold)))

    def invalid_reason(row):
        if pd.isna(row['_dist_direta_servico_escolhido_m']):
            return 'sem_match_servico'
        if row['ligacao_servico_valida']:
            return None
        return 'ratio_excesso_alto'
    merged['motivo_invalidade_servico'] = merged.apply(invalid_reason, axis=1)
    alt_names = []
    alt_cats = []
    alt_dists = []
    serv_points = serv_gdf[['Name', 'Category', 'geometry']].copy()
    for _, row in merged.iterrows():
        origin = row['_origem_pt']
        if not valid_geom(origin):
            alt_names.append(None)
            alt_cats.append(None)
            alt_dists.append(np.nan)
            continue
        d = serv_points.geometry.distance(origin)
        idx = d.idxmin()
        alt_names.append(serv_points.loc[idx, 'Name'])
        alt_cats.append(serv_points.loc[idx, 'Category'])
        alt_dists.append(float(d.loc[idx]))
    merged['servico_candidato_direto'] = alt_names
    merged['categoria_candidato_direto'] = alt_cats
    merged['dist_direta_candidato_m'] = alt_dists
    if replace_invalid:
        use_fallback = ~merged['ligacao_servico_valida'] & merged['dist_direta_candidato_m'].notna() & (merged['dist_direta_candidato_m'] <= max_direct_fallback_m) & (merged['_dist_direta_servico_escolhido_m'].isna() | (merged['dist_direta_candidato_m'] < merged['_dist_direta_servico_escolhido_m'] * 0.6))
        merged['servico_min_id_original'] = merged['servico_min_id']
        merged['servico_min_categoria_original'] = merged['servico_min_categoria']
        merged.loc[use_fallback, 'servico_min_id'] = merged.loc[use_fallback, 'servico_candidato_direto']
        merged.loc[use_fallback, 'servico_min_categoria'] = merged.loc[use_fallback, 'categoria_candidato_direto']
        merged.loc[use_fallback, 'metodo_escolha_servico'] = 'fallback_direto'
        merged.loc[~use_fallback, 'metodo_escolha_servico'] = 'rede'
    else:
        merged['metodo_escolha_servico'] = np.where(merged['ligacao_servico_valida'], 'rede', 'rede_invalida_sinalizada')
    print('Service-data validation failed.')
    print('Total registos:', len(merged))
    n_invalid = int((~merged['ligacao_servico_valida']).sum())
    print('Invalid links:', n_invalid)
    print('Invalid percentage:', round(100 * n_invalid / len(merged), 2) if len(merged) else 0, '%')
    if n_invalid > 0:
        cols_show = ['osm_id', 'servico_min_id', 'servico_min_categoria', 'distancia_minima_servico', '_dist_direta_servico_escolhido_m', '_ratio_rede_direta', '_excesso_rede_vs_direta_m', 'servico_candidato_direto', 'categoria_candidato_direto', 'dist_direta_candidato_m', 'motivo_invalidade_servico']
        display(merged.loc[~merged['ligacao_servico_valida'], cols_show].sort_values(['_ratio_rede_direta', '_excesso_rede_vs_direta_m'], ascending=False).head(20))
    return gpd.GeoDataFrame(merged, geometry='geometry', crs=result_gdf.crs)

In [ ]:
def run_inline_topology_correction(area_name, subarea_geometry, output_name, base_csv, final_csv=None, services_csv=SERVICES_CSV):
    """Workflow validation diagnostic."""
    if final_csv is None:
        final_csv = base_csv
    G_area_base = G_main.copy()
    print(f'Workflow validation diagnostic.{area_name}')
    print('CSV base:', base_csv)
    print('CSV final:', final_csv)
    G_corr, accepted_links_df, area_diag_df, suspects_df, hotspot_counts_df = run_topology_correction_for_area(area_name=area_name, area_csv=base_csv, G_area=G_area_base, services_csv=services_csv)
    print_area_correction_report(area_name=area_name, area_diag_df=area_diag_df, suspects_df=suspects_df, hotspot_counts_df=hotspot_counts_df, accepted_links_df=accepted_links_df, final_result_gdf=None)
    print(f'Workflow validation diagnostic.{area_name}Workflow validation diagnostic.')
    corrected_result = process_subarea_with_graph(subarea_geometry=subarea_geometry, output_name=f'{output_name}_corr', G_graph=G_corr)
    print(f'\n>>> Validar serviço escolhido na área {area_name}...')
    corrected_result = validate_service_selection(result_gdf=corrected_result, services_csv=services_csv, graph_crs=GRAPH_CRS, ratio_threshold=2.5, excess_threshold=150.0, max_direct_fallback_m=80.0, replace_invalid=False)
    print_area_correction_report(area_name=f'{area_name} (após recálculo)', area_diag_df=area_diag_df, suspects_df=suspects_df, hotspot_counts_df=hotspot_counts_df, accepted_links_df=accepted_links_df, final_result_gdf=corrected_result)
    print(f'\n>>> Guardar CSV final corrigido da área {area_name}...')
    save_area_result_csv(corrected_result, final_csv)
    print('Final CSV saved to:', final_csv)
    return {'resultado_corr': corrected_result, 'G_corr': G_corr, 'accepted_links_df': accepted_links_df, 'area_diag_df': area_diag_df, 'suspects_df': suspects_df, 'hotspot_counts_df': hotspot_counts_df, 'base_csv': base_csv, 'final_csv': final_csv}

In [ ]:
def run_topology_correction_for_area(area_name, area_csv, G_area, services_csv=SERVICES_CSV):
    services_gdf = load_services_gdf(services_csv)
    area_diag_df = load_area_results_for_diagnostics(area_csv, services_gdf)
    suspects_df, hotspot_counts_df = detect_hotspots(area_diag_df, min_cases_per_hotspot=MIN_CASES_PER_HOTSPOT)
    G_corr = G_area.copy()
    accepted_all = []
    if len(hotspot_counts_df) == 0:
        print(f'[{area_name}] No hotspot found.')
        accepted_links_df = pd.DataFrame()
        assign_raster_slopes_to_graph(G_corr, only_missing=True)
        return (G_corr, accepted_links_df, area_diag_df, suspects_df, hotspot_counts_df)
    print(f'[{area_name}] Hotspots found: {len(hotspot_counts_df)}')
    for _, hrow in hotspot_counts_df.iterrows():
        hotspot_id = hrow['hotspot_id']
        hotspot_cases = suspects_df[suspects_df['hotspot_id'] == hotspot_id].copy()
        print(f'\n[{area_name}] Hotspot {hotspot_id}Workflow validation diagnostic.{len(hotspot_cases)}')
        G_corr, accepted_df, remaining_cases = iterative_hotspot_fix(G_start=G_corr, cases_df=hotspot_cases)
        if accepted_df is not None and len(accepted_df) > 0:
            accepted_df = accepted_df.copy()
            accepted_df['area_name'] = area_name
            accepted_df['hotspot_id'] = hotspot_id
            accepted_all.append(accepted_df)
    if len(accepted_all) > 0:
        accepted_links_df = pd.concat(accepted_all, ignore_index=True)
    else:
        accepted_links_df = pd.DataFrame()
    assign_raster_slopes_to_graph(G_corr, only_missing=True)
    return (G_corr, accepted_links_df, area_diag_df, suspects_df, hotspot_counts_df)

In [ ]:
corr_northwest = run_inline_topology_correction(area_name='northwest', subarea_geometry=northwest_geom, output_name='northwest_lordelo_ouro_massarelos', base_csv='lordelo_ouro_massarelos_northwest_base.csv', final_csv='lordelo_ouro_massarelos_northwest.csv', services_csv=SERVICES_CSV)
accepted_links_northwest = corr_northwest['accepted_links_df']
area_diag_northwest = corr_northwest['area_diag_df']
suspects_northwest = corr_northwest['suspects_df']
hotspot_counts_northwest = corr_northwest['hotspot_counts_df']
G_northwest_corr = corr_northwest['G_corr']
edificios_northwest_lordelo_ouro_massarelos_resultado_corr = corr_northwest['resultado_corr']

### Northeast subarea

In [ ]:
edificios_northeast_lordelo_ouro_massarelos_resultado = process_subarea(northeast_geom, 'northeast_lordelo_ouro_massarelos')
save_area_result_csv(edificios_northeast_lordelo_ouro_massarelos_resultado, 'lordelo_ouro_massarelos_northeast_base.csv')

In [ ]:
corr_northeast = run_inline_topology_correction(area_name='northeast', subarea_geometry=northeast_geom, output_name='northeast_lordelo_ouro_massarelos', base_csv='lordelo_ouro_massarelos_northeast_base.csv', final_csv='lordelo_ouro_massarelos_northeast.csv', services_csv=SERVICES_CSV)
accepted_links_northeast = corr_northeast['accepted_links_df']
area_diag_northeast = corr_northeast['area_diag_df']
suspects_northeast = corr_northeast['suspects_df']
hotspot_counts_northeast = corr_northeast['hotspot_counts_df']
G_northeast_corr = corr_northeast['G_corr']
edificios_northeast_lordelo_ouro_massarelos_resultado_corr = corr_northeast['resultado_corr']
print('northeast: Corrected final CSV saved to', corr_northeast['final_csv'])
print('northeast: accepted links =', len(accepted_links_northeast))
print('Topology-quality-control diagnostic.', len(hotspot_counts_northeast))
if len(hotspot_counts_northeast) > 0:
    display(hotspot_counts_northeast.head(10))
print('northeast: suspect cases =', len(suspects_northeast))
print('northeast: accepted links =', len(accepted_links_northeast))
if len(accepted_links_northeast) > 0:
    display(accepted_links_northeast)

### Southwest subarea

In [ ]:
edificios_southwest_lordelo_ouro_massarelos_resultado = process_subarea(southwest_geom, 'southwest_lordelo_ouro_massarelos')
save_area_result_csv(edificios_southwest_lordelo_ouro_massarelos_resultado, 'lordelo_ouro_massarelos_southwest_base.csv')

In [ ]:
corr_southwest = run_inline_topology_correction(area_name='southwest', subarea_geometry=southwest_geom, output_name='southwest_lordelo_ouro_massarelos', base_csv='lordelo_ouro_massarelos_southwest_base.csv', final_csv='lordelo_ouro_massarelos_southwest.csv', services_csv=SERVICES_CSV)
accepted_links_southwest = corr_southwest['accepted_links_df']
area_diag_southwest = corr_southwest['area_diag_df']
suspects_southwest = corr_southwest['suspects_df']
hotspot_counts_southwest = corr_southwest['hotspot_counts_df']
G_southwest_corr = corr_southwest['G_corr']
edificios_southwest_lordelo_ouro_massarelos_resultado_corr = corr_southwest['resultado_corr']
print('southwest: Corrected final CSV saved to', corr_southwest['final_csv'])
print('southwest: accepted links =', len(accepted_links_southwest))
print('Topology-quality-control diagnostic.', len(hotspot_counts_southwest))
if len(hotspot_counts_southwest) > 0:
    display(hotspot_counts_southwest.head(10))
print('southwest: suspect cases =', len(suspects_southwest))
print('southwest: accepted links =', len(accepted_links_southwest))
if len(accepted_links_southwest) > 0:
    display(accepted_links_southwest)

### Southeast subarea

In [ ]:
edificios_southeast_lordelo_ouro_massarelos_resultado = process_subarea(southeast_geom, 'southeast_lordelo_ouro_massarelos')
save_area_result_csv(edificios_southeast_lordelo_ouro_massarelos_resultado, 'lordelo_ouro_massarelos_southeast_base.csv')

In [ ]:
corr_southeast = run_inline_topology_correction(area_name='southeast', subarea_geometry=southeast_geom, output_name='southeast_lordelo_ouro_massarelos', base_csv='lordelo_ouro_massarelos_southeast_base.csv', final_csv='lordelo_ouro_massarelos_southeast.csv', services_csv=SERVICES_CSV)
accepted_links_southeast = corr_southeast['accepted_links_df']
area_diag_southeast = corr_southeast['area_diag_df']
suspects_southeast = corr_southeast['suspects_df']
hotspot_counts_southeast = corr_southeast['hotspot_counts_df']
G_southeast_corr = corr_southeast['G_corr']
edificios_southeast_lordelo_ouro_massarelos_resultado_corr = corr_southeast['resultado_corr']
print('southeast: Corrected final CSV saved to', corr_southeast['final_csv'])
print('southeast: accepted links =', len(accepted_links_southeast))
print('Topology-quality-control diagnostic.', len(hotspot_counts_southeast))
if len(hotspot_counts_southeast) > 0:
    display(hotspot_counts_southeast.head(10))
print('southeast: suspect cases =', len(suspects_southeast))
print('southeast: accepted links =', len(accepted_links_southeast))
if len(accepted_links_southeast) > 0:
    display(accepted_links_southeast)

## 7. Merge the four subarea outputs

In [ ]:
import pandas as pd
from pathlib import Path
subarea_files = [f'{PARISH_SLUG}_northwest.csv', f'{PARISH_SLUG}_northeast.csv', f'{PARISH_SLUG}_southwest.csv', f'{PARISH_SLUG}_southeast.csv']
missing_subareas = [name for name in subarea_files if not Path(name).exists()]
if missing_subareas:
    raise FileNotFoundError('Missing corrected subarea files: ' + ', '.join(missing_subareas))
merged_parish_results = pd.concat([pd.read_csv(name, low_memory=False) for name in subarea_files], ignore_index=True)
if merged_parish_results['osm_id'].duplicated().any():
    raise ValueError('Duplicate building IDs were found when merging subareas.')
OUT_BUILDINGS = f'resultados_acessibilidade_800m_{PARISH_SLUG}_global.csv'
merged_parish_results.to_csv(OUT_BUILDINGS, index=False)
od_files = [f'northwest_{PARISH_SLUG}_corr_od800.csv', f'northeast_{PARISH_SLUG}_corr_od800.csv', f'southwest_{PARISH_SLUG}_corr_od800.csv', f'southeast_{PARISH_SLUG}_corr_od800.csv']
if SAVE_OD_LONG:
    missing_od = [name for name in od_files if not Path(name).exists()]
    if missing_od:
        raise FileNotFoundError('Missing corrected OD files: ' + ', '.join(missing_od))
    od_parish = pd.concat([pd.read_csv(name, low_memory=False) for name in od_files], ignore_index=True)
    OUT_OD = f'resultados_od_800m_{PARISH_SLUG}_global.csv'
    od_parish.to_csv(OUT_OD, index=False)
    print('Final OD pairs:', len(od_parish))
print('Final building-level records:', len(merged_parish_results))
print('Building-level output:', PARISH_WORK_DIR / OUT_BUILDINGS)

## 8. Validate the final parish output

In [ ]:
df = pd.read_csv(f'resultados_acessibilidade_800m_{PARISH_SLUG}_global.csv', low_memory=False)
print('Rows:', len(df))
print('Duplicate osm_id:', int(df['osm_id'].duplicated().sum()))
inconsistent_cases = df[(df['numero_servicos_proximos'] == 0) & df['tempo_medio_servicos_seg'].notna()]
print('Zero-destination rows with a non-missing mean time:', len(inconsistent_cases))

## Final validation of the parish building-level output

The final checks verify unique building IDs, required travel-time fields, monotonicity across walking speeds, the fixed 800 m OD limit and consistency between zero-destination buildings and missing travel times. The validated output is then ready for municipal integration.


In [ ]:
scenario_cols = []
for scenario_name in SCENARIO_NAMES:
    scenario_cols.extend([f'tempo_medio_seg__{scenario_name}', f'tempo_min_seg__{scenario_name}'])
required_cols = ['osm_id', 'distancia_media_servicos', 'distancia_minima_servico', 'numero_servicos_proximos', 'tempo_medio_servicos_seg', 'tempo_minimo_servico_seg', *scenario_cols]
missing = [c for c in required_cols if c not in merged_parish_results.columns]
if missing:
    raise KeyError('Missing required output columns: ' + ', '.join(missing))
if merged_parish_results['osm_id'].duplicated().any():
    raise ValueError('Duplicate osm_id values were found.')
for prefix in ['flat', 'tobler']:
    t05 = pd.to_numeric(merged_parish_results[f'tempo_medio_seg__{prefix}_0p5'], errors='coerce')
    t07 = pd.to_numeric(merged_parish_results[f'tempo_medio_seg__{prefix}_0p7'], errors='coerce')
    t09 = pd.to_numeric(merged_parish_results[f'tempo_medio_seg__{prefix}_0p9'], errors='coerce')
    valid = t05.notna() & t07.notna() & t09.notna()
    if not ((t05[valid] >= t07[valid]) & (t07[valid] >= t09[valid])).all():
        raise AssertionError(f'Walking-time monotonicity failed for the {prefix} model.')
no_destination = merged_parish_results['numero_servicos_proximos'].fillna(0).eq(0)
if merged_parish_results.loc[no_destination, [f'tempo_medio_seg__{s}' for s in SCENARIO_NAMES]].notna().any().any():
    raise AssertionError('Buildings without accessible destinations must have missing mean travel times.')
if SAVE_OD_LONG:
    od_check = pd.read_csv(f'resultados_od_800m_{PARISH_SLUG}_global.csv', low_memory=False)
    if pd.to_numeric(od_check['network_distance_m'], errors='coerce').dropna().gt(MAX_NETWORK_DISTANCE_M + 1e-06).any():
        raise AssertionError('An OD pair exceeds the fixed 800 m network catchment.')
print('FINAL PARISH VALIDATION: OK')

## Parish completion status

If all assertions pass, the parish output is considered complete. The municipal PCA/UAI/AAVI analysis and the multiscale spatial-aggregation analysis are performed only after the seven parish datasets are concatenated.


## Raster-directional slope plausibility check

This diagnostic summarises the distribution of slopes used by the directed graph and the resulting Tobler/flat cost ratios. No arbitrary clipping or winsorisation threshold is applied.


In [ ]:
rows_slope = []
for u, v, data in G_main.edges(data=True):
    length = float(data.get('length', np.nan))
    slope = data.get('raster_slope', np.nan)
    if np.isfinite(length) and length > 0 and np.isfinite(slope):
        factor = normalized_tobler_factor(slope)
        rows_slope.append({'length_m': length, 'raster_slope': float(slope), 'slope_pct': float(slope) * 100.0, 'tobler_factor': factor, 'ratio_tobler_flat': 1.0 / max(factor, 1e-12)})
raster_edge_diagnostics = pd.DataFrame(rows_slope)
print('Edges with raster-directional slope:', len(raster_edge_diagnostics))
if len(raster_edge_diagnostics):
    print('Terrain-adjustment validation diagnostic.')
    print(raster_edge_diagnostics['slope_pct'].abs().describe(percentiles=[0.9, 0.95, 0.99, 0.995, 0.999]))
    print('\nTobler / flat cost ratio:')
    print(raster_edge_diagnostics['ratio_tobler_flat'].describe(percentiles=[0.9, 0.95, 0.99, 0.995, 0.999]))
    print('Terrain-adjustment validation diagnostic.', int((raster_edge_diagnostics['slope_pct'].abs() > 100).sum()))
    print('Terrain-adjustment validation diagnostic.', int((raster_edge_diagnostics['ratio_tobler_flat'] > 100).sum()))

In [ ]:
os.chdir(REPO_ROOT)
print("Working directory restored:", REPO_ROOT)
